# 11주차: 데이터 결합 및 그룹 연산

## 1. 데이터프레임 연결

### 1-1. `concat()` 함수

- 서로 다른 데이터프레임의 구조와 속성이 동일하다면 행(`axis=0`, 기본값) 또는 열(`axis=1`) 방향으로 연결하여 일관된 데이터 구성을 유지할 수 있음
- `concat()` 함수는 여러 개의 데이터프레임을 리스트 형태로 전달받아 하나의 데이터프레임으로 연결함
- `join='outer'` 옵션(기본값)은 열 이름이 서로 다른 경우에는 합집합 기준으로 병합되며, `join='inner'` 옵션을 사용하면 공통으로 존재하는 열만 교집합 기준으로 병합됨

- 두 데이터프레임을 연결하면 열 이름의 합집합(outer join)을 기준으로 새로운 열 구성이 생성됨
- 첫 번째 데이터프레임에는 `d` 열이 없기 때문에 해당 위치에는 자동으로 `NaN` 값이 입력됨

In [1]:
import numpy as np
import pandas as pd

# 첫 번째 데이터프레임 (열: a, b, c)
df1 = pd.DataFrame(
    np.arange(12).reshape(4, 3),
    columns=["a", "b", "c"]
)

# 두 번째 데이터프레임 (열: a, b, c, d)
df2 = pd.DataFrame(
    np.arange(16).reshape(4, 4),
    columns=["a", "b", "c", "d"],
    index=[2, 3, 4, 5]  # 인덱스 지정
)

In [2]:
print("첫 번째 데이터프레임:\n", df1)
print("\n두 번째 데이터프레임:\n", df2)

# 두 데이터프레임 연결
result = pd.concat([df1, df2])
print(result)

첫 번째 데이터프레임:
    a   b   c
0  0   1   2
1  3   4   5
2  6   7   8
3  9  10  11

두 번째 데이터프레임:
     a   b   c   d
2   0   1   2   3
3   4   5   6   7
4   8   9  10  11
5  12  13  14  15
    a   b   c     d
0   0   1   2   NaN
1   3   4   5   NaN
2   6   7   8   NaN
3   9  10  11   NaN
2   0   1   2   3.0
3   4   5   6   7.0
4   8   9  10  11.0
5  12  13  14  15.0


- 데이터프레임을 열 방향으로 연결하려면 `axis=1` 옵션을 지정해야 함
- 행 인덱스는 기본적으로 `join='outer'` 옵션이 적용되어 각 데이터프레임의 행 인덱스의 합집합 기준으로 결합되며, 공통되지 않는 인덱스의 위치에는 `NaN` 값이 자동으로 채워짐

In [3]:
# 열 방향 연결 (axis=1)
result = pd.concat([df1, df2], axis=1)
print(result)

     a     b     c     a     b     c     d
0  0.0   1.0   2.0   NaN   NaN   NaN   NaN
1  3.0   4.0   5.0   NaN   NaN   NaN   NaN
2  6.0   7.0   8.0   0.0   1.0   2.0   3.0
3  9.0  10.0  11.0   4.0   5.0   6.0   7.0
4  NaN   NaN   NaN   8.0   9.0  10.0  11.0
5  NaN   NaN   NaN  12.0  13.0  14.0  15.0


- `join='inner'` 옵션을 설정하면 연결할 데이터프레임의 행 인덱스 교집합을 기준으로 결합함
- 두 데이터프레임에 공통으로 존재하는 인덱스만 유지되고 나머지 인덱스의 행은 제외됨

In [4]:
# 행 인덱스 교집합 기준으로 열 방향 결합
result = pd.concat([df1, df2], axis=1, join='inner')
print(result)

   a   b   c  a  b  c  d
2  6   7   8  0  1  2  3
3  9  10  11  4  5  6  7


### 1-2. `merge()` 함수

- `concat()` 함수가 단순히 여러 데이터프레임을 위아래 또는 좌우로 연결하는 것이라면 `merge()` 함수는 특정 기준에 따라 두 데이터프레임을 병합함
- `on=None`: 두 데이터프레임에 공통으로 존재하는 모든 열을 기준으로 병합함
- `how='inner'`(기본값): 두 데이터프레임에 공통으로 존재하는 교집합을 기준으로 병합함

In [5]:
# 첫 번째 데이터프레임
df1 = pd.DataFrame({
    "key": ["b", "b", "a", "c", "a", "a", "b"],
    "data1": range(7)
})

# 두 번째 데이터프레임
df2 = pd.DataFrame({
    "key": ["a", "b", "d"],
    "data2": range(3)
})

In [6]:
print("첫 번째 데이터프레임:\n", df1)
print("\n두 번째 데이터프레임:\n", df2)

# 공통 열(key)을 기준으로 병합
result = pd.merge(df2, df1)
print(result)

첫 번째 데이터프레임:
   key  data1
0   b      0
1   b      1
2   a      2
3   c      3
4   a      4
5   a      5
6   b      6

두 번째 데이터프레임:
   key  data2
0   a      0
1   b      1
2   d      2
  key  data2  data1
0   a      0      2
1   a      0      4
2   a      0      5
3   b      1      0
4   b      1      1
5   b      1      6


- 이번 예제에서는 `on`과 `how` 옵션을 함께 사용하여 병합함
- `on='key'`: 병합 시 기준이 되는 공통 열 이름(`key`)을 지정함
- `how='outer'`: 기준이 되는 열 값이 양쪽 데이터프레임 중 어느 한쪽에만 존재하더라도 모두 포함하며, 한쪽에 값이 없으면 해당 부분은 `NaN`으로 채워짐

In [7]:
# 'key' 열 기준으로 외부 병합 (합집합)
result = pd.merge(df1, df2, how="outer", on="key")
print(result)

  key  data1  data2
0   a    2.0    0.0
1   a    4.0    0.0
2   a    5.0    0.0
3   b    0.0    1.0
4   b    1.0    1.0
5   b    6.0    1.0
6   c    3.0    NaN
7   d    NaN    2.0


- 두 데이터프레임에 기준 열 이름이 서로 다르거나 중복되는 열 이름이 존재할 경우에는 병합 기준 열을 각각 따로 지정할 수 있음
- `left_on`과 `right_on` 옵션은 각각 왼쪽과 오른쪽 데이터프레임의 기준 열을 의미함
- 예를 들어, `how='right'` 옵션은 오른쪽 데이터프레임의 열을 기준으로 병합하는 것을 의미함
- 즉, 오른쪽 데이터프레임에 존재하는 모든 열 값이 결과에 포함되며, 왼쪽 데이터프레임에 해당 열 값이 없으면 `NaN`으로 채워짐

In [8]:
# 왼쪽 데이터프레임
df1 = pd.DataFrame({
    "key_left": ["b", "b", "a", "c", "a", "a", "b"],
    "data1": range(7)
})

# 오른쪽 데이터프레임
df2 = pd.DataFrame({
    "key_right": ["a", "b", "d"],
    "data2": range(3)
})

In [9]:
print("왼쪽 데이터프레임:\n", df1)
print("\n오른쪽 데이터프레임:\n", df2)

# 서로 다른 키 이름을 기준으로 병합 (오른쪽 기준)
result = pd.merge(df1, df2, left_on="key_left", right_on="key_right", how="right")
print(result)

왼쪽 데이터프레임:
   key_left  data1
0        b      0
1        b      1
2        a      2
3        c      3
4        a      4
5        a      5
6        b      6

오른쪽 데이터프레임:
   key_right  data2
0         a      0
1         b      1
2         d      2
  key_left  data1 key_right  data2
0        a    2.0         a      0
1        a    4.0         a      0
2        a    5.0         a      0
3        b    0.0         b      1
4        b    1.0         b      1
5        b    6.0         b      1
6      NaN    NaN         d      2


## 2. 그룹 연산

### 2-1. 그룹 연산 개요

- 복잡한 데이터를 특정 기준에 따라 여러 그룹으로 나누어 관찰하는 것은 데이터 분석에서 매우 유용한 방법임
- 이러한 과정은 데이터를 여러 하위 집단으로 분류하여 각 그룹의 통계치를 계산하는 그룹 연산이라고 함
- 그룹 연산은 데이터를 집계, 변환, 필터링하는 데 효율적이며, 일반적으로 다음의 3단계 과정으로 이루어짐
  - 1단계(split): 데이터를 특정 기준에 따라 그룹으로 분할함
  - 2단계(apply): 각 그룹에 대해 합계, 평균, 최대값 등의 집계 함수나 변환 혹은 필터링 메서드를 적용함
  - 3단계(combine): 2단계에서 얻은 결과를 하나의 데이터로 결합함

### 2-2. 분할 단계

- `groupby()` 함수는 데이터프레임의 특정 열을 기준으로 데이터를 그룹화하여 그룹 객체를 반환함
  - 기준이 되는 열은 1개 또는 여러 개를 리스트 형태로 지정 가능함
  - 반환된 그룹 객체는 이후 집계, 변환, 필터링 등의 연산에 활용 가능함

In [10]:
# ! pip install seaborn  # 필요시 주석 해제하여 설치
import seaborn as sns

# 타이타닉 데이터 불러오기
titanic = sns.load_dataset("titanic")

# 필요한 열만 선택
df = titanic.loc[:, ["age", "sex", "class", "fare", "survived"]]

# 데이터 미리보기
print(df.head())

# 'class' 열을 기준으로 그룹화
grouped = df.groupby("class")
print(grouped)

    age     sex  class     fare  survived
0  22.0    male  Third   7.2500         0
1  38.0  female  First  71.2833         1
2  26.0  female  Third   7.9250         1
3  35.0  female  First  53.1000         1
4  35.0    male  Third   8.0500         0


- 앞에서 생성한 그룹 객체에 다양한 연산 함수를 적용할 수 있음
  - 다음 예시는 그룹 객체에 포함된 3개 그룹에 대해 평균값을 구하기 위해 `mean()` 함수를 적용함
  - `numeric_only=True` 옵션은 연산이 가능한 수치형 열에 대해서만 계산이 수행되며 문자형 열은 자동으로 제외됨

In [11]:
# class 열을 기준으로 그룹화 후 평균 계산
group_mean = df.groupby("class").mean(numeric_only=True)
print(group_mean)

              age       fare  survived
class                                 
First   38.233441  84.154687  0.629630
Second  29.877630  20.662183  0.472826
Third   25.140620  13.675550  0.242363


- 그룹 객체에 `get_group()` 함수를 적용하면 특정 그룹의 데이터만 선택하여 추출할 수 있음
  - 예를 들어, `class`가 `Third`인 승객만 따로 선택하여 확인 가능하며, 이를 통해 그룹별 세부 데이터에 접근하거나 개별 분석을 수행할 수 있음

In [12]:
# 'Third' 클래스에 해당하는 그룹만 선택하여 상위 5개 행 출력
third_class = grouped.get_group("Third").head()
print(third_class)

    age     sex  class     fare  survived
0  22.0    male  Third   7.2500         0
2  26.0  female  Third   7.9250         1
4  35.0    male  Third   8.0500         0
5   NaN    male  Third   8.4583         0
7   2.0    male  Third  21.0750         0


- `groupby()` 함수는 하나의 열뿐만 아니라 두 개 이상의 열을 동시에 기준으로 그룹화할 수 있음
  - 다음 예시는 `class`와 `sex` 두 열을 기준으로 그룹화를 수행하여 각 그룹별 평균값을 계산함
  - 그룹화 결과는 두 기준 열이 계층적 인덱스(MultiIndex) 형태로 지정되어 표시됨

In [13]:
# 'class'와 'sex'를 기준으로 그룹화 후 평균 계산
group_mean_multi = df.groupby(["class", "sex"]).mean(numeric_only=True)

print(group_mean_multi)

                     age        fare  survived
class  sex                                    
First  female  34.611765  106.125798  0.968085
       male    41.281386   67.226127  0.368852
Second female  28.722973   21.970121  0.921053
       male    30.740707   19.741782  0.157407
Third  female  21.750000   16.118810  0.500000
       male    26.507589   12.661633  0.135447


- 여러 열을 기준으로 그룹화한 후 특정 그룹만 선택적으로 추출할 때 `get_group()` 함수를 사용할 수 있음
  - 이때 인자(key)는 튜플(tuple) 형태로 전달해야 함
  - 아래는 그룹 객체의 6개 그룹 중에서 `('Third', 'female')` 그룹만 추출한 예시임

In [14]:
# ('Third', 'female') 그룹만 추출하여 상위 5개 행 출력
third_female = df.groupby(["class", "sex"]).get_group(("Third", "female")).head()
print(third_female)

     age     sex  class     fare  survived
2   26.0  female  Third   7.9250         1
8   27.0  female  Third  11.1333         1
10   4.0  female  Third  16.7000         1
14  14.0  female  Third   7.8542         0
18  31.0  female  Third  18.0000         0


### 2-3. 적용/결합 단계

- 앞서 분할된 그룹 객체에 대해 평균, 최대값, 최소값 등 다양한 연산을 적용할 수 있으며 이러한 과정을 데이터 집계(aggregation)라고 함
  - pandas는 집계 기능을 수행할 수 있는 다양한 기본 함수를 제공함
  - 아래 예시는 `std()` 함수를 사용하여 각 `class` 그룹의 표준편차를 계산한 결과임

In [15]:
# class별 각 수치형 변수의 표준편차 계산
group_std = df.groupby("class").std(numeric_only=True)
print(group_std)

# 특정 열(fare)만 선택하여 표준편차 계산
fare_std = df.groupby("class")["fare"].std()
print(fare_std)

              age       fare  survived
class                                 
First   14.802856  78.380373  0.484026
Second  14.001077  13.417399  0.500623
Third   12.495398  11.778142  0.428949
class
First     78.380373
Second    13.417399
Third     11.778142
Name: fare, dtype: float64


- `agg()` 함수는 그룹 객체에 사용자 정의 함수를 적용할 수 있게 함
  - 내장된 집계 함수(예: `mean()`, `sum()`, `std()` 등) 외에도 사용자가 직접 정의한 함수를 전달 가능함
  - 예를 들어, 각 그룹에서 최대값과 최소값의 차이를 계산하려면 사용자 정의 함수를 만들어 `agg()`에 적용함

In [16]:
# 사용자 정의 함수: 최댓값과 최솟값의 차이 계산
def min_max(x):
    return x.max() - x.min()


# class별로 사용자 정의 함수를 적용하여 집계 (수치형 열만 선택)
result = df.groupby("class")[["age", "fare", "survived"]].agg(min_max)
print(result)

          age      fare  survived
class                            
First   79.08  512.3292         1
Second  69.33   73.5000         1
Third   73.58   69.5500         1


- 여러 개의 함수를 동시에 사용하여 각 그룹별 데이터에 대해 다양한 집계 연산을 수행할 수 있음
  - 각 열에 동일한 함수를 여러 개 적용하려면 리스트(list) 형태로 함수명을 전달함
  - 열마다 서로 다른 함수를 적용하려면 딕셔너리 형태로 전달하며, 이러한 방식을 통해 그룹별로 다양한 통계치를 한 번에 계산할 수 있음

In [17]:
# 여러 함수를 리스트로 전달 (수치형 열만 선택)
result1 = df.groupby("class")[["age", "fare", "survived"]].agg(["min", "max"])
print(result1)

         age       fare           survived    
         min   max  min       max      min max
class                                         
First   0.92  80.0  0.0  512.3292        0   1
Second  0.67  70.0  0.0   73.5000        0   1
Third   0.42  74.0  0.0   69.5500        0   1


In [18]:
# 열마다 다른 함수를 적용 (딕셔너리 형태로 전달)
result2 = df.groupby("class").agg({
    "fare": ["min", "max"],
    "age": "mean"
})
print(result2)

       fare                  age
        min       max       mean
class                           
First   0.0  512.3292  38.233441
Second  0.0   73.5000  29.877630
Third   0.0   69.5500  25.140620


- 앞서 살펴본 `agg()` 함수는 각 그룹에 대해 하나의 결과값(집계 결과)을 반환함
- 반면 `transform()` 함수는 그룹별 연산을 수행하여 연산 결과를 기존 데이터프레임의 행 인덱스와 동일한 구조로 반환함
- 즉, 그룹별 연산 결과를 원래 데이터프레임과 같은 크기로 확장하여 정렬하며, 이를 통해 그룹 연산 결과를 기존 데이터프레임에 바로 추가하거나 비교할 수 있음

In [19]:
# class별로 age에 대해 transform 적용
result = df.groupby("class")["age"].transform(min_max)
print(result.head(10))

0    73.58
1    79.08
2    73.58
3    79.08
4    73.58
5    73.58
6    79.08
7    73.58
8    73.58
9    69.33
Name: age, dtype: float64


- `filter()` 함수는 그룹 객체에 조건식을 가진 함수를 전달하여 조건을 만족하는 그룹만 선택해 반환함
  - 그룹 단위로 조건을 평가하여 `True`인 그룹만 남기고 나머지는 제거하며, 필터링 조건은 `lambda` 함수나 사용자 정의 함수를 통해 지정할 수 있음

In [20]:
# 그룹의 데이터 개수가 200개 이상인 그룹만 선택
result1 = df.groupby("class").filter(lambda x: len(x) >= 200)
print(result1.head())

# 평균 나이가 30보다 작은 그룹만 선택
result2 = df.groupby("class").filter(lambda x: x["age"].mean() < 30)
print(result2.head())

    age     sex  class     fare  survived
0  22.0    male  Third   7.2500         0
1  38.0  female  First  71.2833         1
2  26.0  female  Third   7.9250         1
3  35.0  female  First  53.1000         1
4  35.0    male  Third   8.0500         0
    age     sex  class     fare  survived
0  22.0    male  Third   7.2500         0
2  26.0  female  Third   7.9250         1
4  35.0    male  Third   8.0500         0
5   NaN    male  Third   8.4583         0
7   2.0    male  Third  21.0750         0


- `apply()` 함수는 pandas 그룹 객체의 각 그룹에 사용자 정의 함수를 적용할 때 사용함
  - 그룹 단위로 데이터를 전달받아 특정 연산을 수행하고 결과를 결합하여 반환함
  - `transform()` 함수와 다르게 그룹별 전체 결과를 하나의 시리즈나 데이터프레임으로 반환할 수 있음

In [21]:
# 사용자 정의 함수: Z-score 계산
def z_score(x):
    return (x - x.mean()) / x.std()


# class별로 age 데이터에 z-score 함수 적용
result = df.groupby("class")["age"].apply(z_score)
print(result.head())

class    
First  1    -0.015770
       3    -0.218434
       6     1.065103
       11    1.335321
       23   -0.691315
Name: age, dtype: float64
